# 14 — Federated FLAIR prediction from T1 + T2 (Round-2, Phase 3)

**What.** Train **IFCNN** (output zeroed outside the input-derived brain mask `(T1>0)|(T2>0)`; same backbone for every arm; review M5) to predict the held-out **FLAIR** from **T1 + T2** with **FedAvg** over the 3 grade-skewed clients, plus a **centralised** reference trained on the pooled client-train patients with the identical trainer (K = 1). **5 seeds** each. Final metrics on the **54 test patients** use exactly the Phase-2 baseline definitions, so the model is compared like-for-like with `zero`, `T2`, `mean(T1,T2)` and `linear(T1,T2)`.

**Protocol (fixed in advance; no model selection).** T = 30 rounds, E = 1 local epoch of 1 000 random slices per client (centralised: 3 000 slices per round = same total samples), batch 16, Adam lr 1e-3, loss L1 + (1 − SSIM) against FLAIR. Validation patients are used only for the printed learning curve; the round-30 model is always the one evaluated.

**Needs a GPU runtime** (Runtime → Change runtime type → T4 GPU).

**Runtime.** Copy ≈ 5 min; sanity gate ≈ 3 min; each run ≈ 15–20 min on a T4 → 10 runs ≈ 3 h. **Resumable per run**: finished (method, seed) runs are skipped, so if the session dies just *Run all* again. A run interrupted mid-way restarts from round 1 (≤ 20 min lost).

**Outputs (Drive `MyDrive/FedSafeFuse/`):**
| Drive path | Download to local |
|---|---|
| `checkpoints/flair_<method>_seed<s>.pt` | (stays on Drive; Phase 4 uses them) |
| `results/flair_runs/<method>_seed<s>_history.csv` | — |
| `results/flair_runs/<method>_seed<s>_test.csv` | — |
| `results/flair_test_summary.csv` | `results/flair_test_summary.csv` |
| `results/flair_train_history.csv` | `results/flair_train_history.csv` |
| `figures/phase3_flair_examples.png` | `figures/phase3_flair_examples.png` |

In [ ]:
# 0. Config  (do not change between sessions, or resumed runs will not be comparable)
SEEDS      = [42, 1, 2, 3, 4]
METHODS    = ['fedavg', 'central']
T_ROUNDS   = 30
E_LOCAL    = 1
SAMPLES_PER_CLIENT = 1000      # per local epoch; centralised uses K * this per round
BATCH      = 16
LR         = 1e-3
LOG_EVERY  = 5
GATE_ROUNDS = 5                # sanity gate: short centralised run before the long runs
REQUIRE_GPU = True

In [ ]:
# 1. Setup + mount Drive
import os, sys, json, time, glob, random, shutil, importlib
import numpy as np, pandas as pd
import torch
import matplotlib.pyplot as plt

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch', torch.__version__, '| device', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL      = '/content/local_flair'
SRC_DIR    = '/content/fedsrc'
RUN_DIR    = f'{DRIVE_ROOT}/results/flair_runs'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
for d in [LOCAL, SRC_DIR, RUN_DIR, CKPT_DIR, f'{DRIVE_ROOT}/figures']:
    os.makedirs(d, exist_ok=True)

def atomic_csv(df, path):
    tmp = path + '.tmp'; df.to_csv(tmp, index=False); os.replace(tmp, path)

def atomic_torch_save(obj, path):
    tmp = path + '.tmp'; torch.save(obj, tmp); os.replace(tmp, path)
print('OK')

Embedded verbatim from `src/` (unit-tested locally; do not edit here).

In [ ]:
%%writefile /content/fedsrc/ifcnn.py
"""IFCNN baseline (Zhang et al. 2021).

A general image fusion framework based on a two-stream CNN with element-wise fusion
followed by reconstruction. Simplified from the original paper for our 128x128
single-channel fusion setting. Used as Baseline 2 in the FedSafe-Fuse evaluation.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class IFCNN(nn.Module):
    """Two-stream conv + element-wise fusion + conv reconstruction."""

    FUSION_MODES = ("max", "mean", "sum")

    def __init__(self, base_ch: int = 64, fusion: str = "max"):
        super().__init__()
        assert fusion in self.FUSION_MODES, f"fusion must be one of {self.FUSION_MODES}"
        self.fusion_mode = fusion
        self.stream1 = self._make_stream(base_ch)
        self.stream2 = self._make_stream(base_ch)
        self.recon = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=1),
            nn.BatchNorm2d(base_ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch // 2, 1, kernel_size=1),
        )

    @staticmethod
    def _make_stream(c: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(1, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        f1 = self.stream1(x1)
        f2 = self.stream2(x2)
        if self.fusion_mode == "max":
            fused = torch.maximum(f1, f2)
        elif self.fusion_mode == "mean":
            fused = 0.5 * (f1 + f2)
        else:
            fused = f1 + f2
        return torch.sigmoid(self.recon(fused))


class InputMasked(nn.Module):
    """Zero a fuser's output outside the input-derived brain mask (T1 > 0) | (T2 > 0).

    The mask uses the inputs only, so it is available at inference. Without it a
    sigmoid output is never exactly 0 on the (majority) air background, which
    the trivial baselines (T2, mean(T1, T2)) get for free.
    """

    def __init__(self, net: nn.Module):
        super().__init__()
        self.net = net

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        mask = ((x1 > 0) | (x2 > 0)).to(x1.dtype)
        return self.net(x1, x2) * mask


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


if __name__ == "__main__":
    m = IFCNN()
    n = count_parameters(m)
    print(f"IFCNN trainable params: {n:,} ({n / 1e6:.2f}M)")
    x = torch.randn(2, 1, 128, 128)
    y = m(x, x)
    print(f"Input: {x.shape} -> Output: {y.shape}")

In [ ]:
%%writefile /content/fedsrc/losses.py
"""Composite fusion loss: L1 + beta * (1 - SSIM).

Implements a differentiable SSIM via a Gaussian-windowed convolution
(no external dependency on pytorch_msssim). Designed for single-channel
medical images in [0, 1].
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def _gaussian_window(window_size: int, sigma: float, channels: int) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - (window_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w_2d = g.unsqueeze(0) * g.unsqueeze(1)  # (W, W)
    return w_2d.expand(channels, 1, window_size, window_size).contiguous()


class SSIMLoss(nn.Module):
    """1 - SSIM, computed with a Gaussian sliding window. Single-channel input in [0, 1]."""

    def __init__(
        self,
        window_size: int = 11,
        sigma: float = 1.5,
        data_range: float = 1.0,
        channels: int = 1,
    ):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.C1 = (0.01 * data_range) ** 2
        self.C2 = (0.03 * data_range) ** 2
        self.register_buffer("window", _gaussian_window(window_size, sigma, channels))

    def _filter(self, x: torch.Tensor) -> torch.Tensor:
        return F.conv2d(x, self.window, padding=self.window_size // 2, groups=x.shape[1])

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        mu_p = self._filter(pred)
        mu_t = self._filter(target)
        mu_p2 = mu_p ** 2
        mu_t2 = mu_t ** 2
        mu_pt = mu_p * mu_t
        sigma_p2 = self._filter(pred * pred) - mu_p2
        sigma_t2 = self._filter(target * target) - mu_t2
        sigma_pt = self._filter(pred * target) - mu_pt
        num = (2 * mu_pt + self.C1) * (2 * sigma_pt + self.C2)
        den = (mu_p2 + mu_t2 + self.C1) * (sigma_p2 + sigma_t2 + self.C2)
        ssim_map = num / den
        return 1.0 - ssim_map.mean()


class FusionLoss(nn.Module):
    """L1(fused, ref) + beta * SSIMLoss(fused, ref).

    Reference composite is a per-pixel weighted average of source modalities, per the proposal.
    Default weights are (0.5, 0.5). Override `reference_weights` per dataset if needed.
    """

    def __init__(self, beta: float = 1.0, reference_weights: tuple[float, float] = (0.5, 0.5)):
        super().__init__()
        assert abs(sum(reference_weights) - 1.0) < 1e-6, "reference_weights must sum to 1.0"
        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.beta = beta
        w1, w2 = reference_weights
        self.register_buffer("w1", torch.tensor(w1))
        self.register_buffer("w2", torch.tensor(w2))

    def reference(self, src1: torch.Tensor, src2: torch.Tensor) -> torch.Tensor:
        return self.w1 * src1 + self.w2 * src2

    def forward(
        self,
        fused: torch.Tensor,
        src1: torch.Tensor,
        src2: torch.Tensor,
    ) -> torch.Tensor:
        ref = self.reference(src1, src2)
        return self.l1(fused, ref) + self.beta * self.ssim(fused, ref)


class TargetLoss(nn.Module):
    """L1(pred, target) + beta * SSIMLoss(pred, target) against a held-out target.

    Used when the target (e.g. FLAIR) is a separate acquisition that is not a
    function of the inputs, so the residual |pred - target| is not observable
    at inference.
    """

    def __init__(self, beta: float = 1.0):
        super().__init__()
        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.beta = beta

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return self.l1(pred, target) + self.beta * self.ssim(pred, target)


@torch.no_grad()
def ssim_score(pred: torch.Tensor, target: torch.Tensor) -> float:
    """SSIM (higher is better) for evaluation. Mirrors SSIMLoss without the 1 - ... wrap."""
    loss_fn = SSIMLoss().to(pred.device)
    return float(1.0 - loss_fn(pred, target).item())


@torch.no_grad()
def psnr_score(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> float:
    """PSNR in dB."""
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return float("inf")
    return 10.0 * torch.log10(torch.tensor(data_range ** 2 / mse)).item()

In [ ]:
%%writefile /content/fedsrc/trainer.py
"""In-process FedAvg simulator supporting standard, DP-SGD, and FIPCA modes.

A single-process replacement for Flower's SimulationEngine, designed for Colab
Free's compute budget. Mirrors the round structure of Flower so the algorithm
can be swapped to Flower without changing the experimental protocol.

Three modes:
    - 'standard': vanilla FedAvg (raw weight deltas).
    - 'dpsgd'   : DEPRECATED, NOT DP-SGD. Clips the batch-mean gradient and adds
                  N(0, (sigma*C)^2) to it. Abadi et al. require per-example
                  clipping; this mechanism has sensitivity 2C and no valid
                  (epsilon, delta). Kept only to reproduce archived Round-1/2
                  numbers; do not use for new results (use Opacus).
    - 'fipca'   : (FROP) Client parameter delta is projected to a rank-d subspace via a
                  pseudo-random Gaussian matrix derived from a fixed seed (shared
                  basis across clients, never transmitted). Server averages the
                  rank-d coefficients and reconstructs. With error_feedback=True
                  each client additionally keeps a local residual of the part of
                  its delta the projection discarded and adds it back into the
                  next round's delta before projecting (memory-compensated
                  compression, Stich et al. 2018). The transmitted payload is
                  still exactly rank-d coefficients and the residual never
                  leaves the client.

Communication cost per round is logged in bytes.

FROP projects trainable parameters only. Buffers (BatchNorm running_mean/var,
num_batches_tracked) are averaged uncompressed and counted in bytes_per_round.
Earlier versions projected the whole state_dict, which froze BN statistics at
initialisation (reconstructed buffer deltas ~sqrt(d/N) of the truth, integer
counters truncated to 0); archived FROP results carry that bug.

If ``supervised_target=True`` each batch is (src1, src2, target, ...) and the
loss / eval metrics are computed against the held-out ``target`` instead of
the input-derived reference composite.
"""

from __future__ import annotations

import time
from typing import Callable, Optional

import warnings

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset


def _flatten_state(state: dict) -> torch.Tensor:
    """Concatenate all tensors in a state_dict into a single 1-D float32 vector."""
    return torch.cat([v.flatten().float() for v in state.values()])


def _unflatten_to_state(flat: torch.Tensor, template: dict) -> dict:
    """Inverse of _flatten_state, using template state_dict for shapes."""
    out: dict = {}
    idx = 0
    for k, v in template.items():
        n = v.numel()
        out[k] = flat[idx : idx + n].reshape(v.shape).to(v.dtype)
        idx += n
    return out


class FedAvgTrainer:
    """Single-process federated trainer.

    Parameters
    ----------
    build_model_fn :
        Zero-arg callable that returns a fresh model on CPU. Cloned per client.
    loss_fn_factory :
        Zero-arg callable returning a loss module (e.g. FusionLoss(beta)).
        Recreated per client to avoid sharing buffers (SSIM Gaussian window).
    client_datasets :
        List of K Dataset objects, one per client. Items must be (mri, pet, ...).
    eval_dataset :
        Held-out evaluation set (e.g. the calibration split).
    device :
        'cuda' or 'cpu'.
    mode :
        'standard', 'dpsgd', or 'fipca'.
    samples_per_local_epoch :
        Cap on examples drawn per client per local epoch (Round 1 compute cut).
    fipca_rank :
        d, the rank of the FIPCA random-projection subspace.
    error_feedback :
        FIPCA only. If True, each client accumulates the projection residual
        locally and adds it to the next round's delta before compressing.
        NOTE: with a fixed basis this is provably inert — the residual lies in
        the null space of P, so P @ residual = 0 and the payload is unchanged.
        Error feedback only transmits accumulated information when the basis
        changes between rounds; combine with resample_basis=True.
    resample_basis :
        FIPCA only. If True, the orthonormal basis P is regenerated at the
        start of every round from a per-round seed (seed + 7919 * t). The seed
        schedule is deterministic and shared by all clients, so the basis is
        never transmitted and bytes/round are unchanged. Successive subspaces
        are independent, so the error-feedback residual from round t overlaps
        round t+1's subspace and is gradually transmitted.
    dp_clip, dp_sigma :
        DP-SGD per-batch gradient L2-clip threshold and noise multiplier.
    lr :
        Adam learning rate.
    batch_size :
        Per-client batch size.
    seed :
        Random seed for reproducibility.
    """

    VALID_MODES = ("standard", "dpsgd", "fipca")

    def __init__(
        self,
        build_model_fn: Callable[[], nn.Module],
        loss_fn_factory: Callable[[], nn.Module],
        client_datasets: list,
        eval_dataset: Dataset,
        device: str = "cuda",
        mode: str = "standard",
        samples_per_local_epoch: int = 100,
        fipca_rank: int = 32,
        error_feedback: bool = False,
        resample_basis: bool = False,
        dp_clip: float = 1.0,
        dp_sigma: float = 0.5,
        lr: float = 1e-3,
        batch_size: int = 16,
        seed: int = 42,
        supervised_target: bool = False,
    ):
        assert mode in self.VALID_MODES, f"mode must be one of {self.VALID_MODES}"
        self.build_model_fn = build_model_fn
        self.loss_fn_factory = loss_fn_factory
        self.client_datasets = client_datasets
        self.eval_dataset = eval_dataset
        self.device = device
        self.mode = mode
        self.samples_per_local_epoch = samples_per_local_epoch
        self.fipca_rank = fipca_rank
        self.error_feedback = error_feedback and mode == "fipca"
        self.resample_basis = resample_basis and mode == "fipca"
        self.dp_clip = dp_clip
        self.dp_sigma = dp_sigma
        self.lr = lr
        self.batch_size = batch_size
        self.seed = seed
        self.supervised_target = supervised_target
        if mode == "dpsgd":
            warnings.warn(
                "mode='dpsgd' clips the batch-mean gradient; it is NOT DP-SGD and has "
                "no valid (epsilon, delta). Use only to reproduce archived results.",
                stacklevel=2,
            )
        self._rng = np.random.default_rng(seed)

        # Global model (server's authoritative copy)
        self.global_model = build_model_fn().to(device)
        self.global_state = {k: v.detach().clone() for k, v in self.global_model.state_dict().items()}
        param_names = {n for n, _ in self.global_model.named_parameters()}
        self._param_keys = [k for k in self.global_state if k in param_names]
        self._buffer_keys = [k for k in self.global_state if k not in param_names]

        # FIPCA: deterministic ORTHONORMAL projection matrix P in (rank, n_params).
        # Built via QR decomposition so that P @ P.T = I_rank exactly, which makes
        # P.T @ P a true rank-d orthogonal projector with operator norm 1.
        # Earlier scaled-Gaussian attempt had operator norm ~sqrt(n_params/rank),
        # causing reconstructed deltas to blow up and Adam to diverge (NaN).
        if mode == "fipca":
            self._n_params = sum(self.global_state[k].numel() for k in self._param_keys)
            self._build_basis(seed)
            # Error-feedback residuals: one (n_params,) vector per client, kept
            # strictly client-side. Allocated lazily on first compression.
            self._ef_residuals: list = [None] * len(client_datasets)

    def _build_basis(self, basis_seed: int) -> None:
        """(Re)build the rank-d orthonormal projection P from a seed."""
        gen = torch.Generator(device=self.device).manual_seed(basis_seed)
        G = torch.randn(self._n_params, self.fipca_rank, device=self.device, generator=gen)
        Q, _ = torch.linalg.qr(G)  # Q has orthonormal columns: (n_params, rank)
        self.P = Q.T.contiguous()  # (rank, n_params) with orthonormal rows
        del G, Q

    # ------------------------------------------------------------------
    # Client-side local training
    # ------------------------------------------------------------------
    def _local_subset_loader(self, client_idx: int) -> DataLoader:
        ds = self.client_datasets[client_idx]
        n = min(self.samples_per_local_epoch, len(ds))
        sel = self._rng.choice(len(ds), size=n, replace=False)
        subset = Subset(ds, sel.tolist())
        return DataLoader(subset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def _train_client(self, client_idx: int, n_local_epochs: int) -> dict:
        """Run E local epochs on a fresh random subset of this client's data, return weight delta."""
        local_model = self.build_model_fn().to(self.device)
        local_model.load_state_dict(self.global_state)
        opt = torch.optim.Adam(local_model.parameters(), lr=self.lr, weight_decay=1e-4)
        loss_fn = self.loss_fn_factory().to(self.device)

        local_model.train()
        for _ in range(n_local_epochs):
            loader = self._local_subset_loader(client_idx)
            for batch in loader:
                mri, pet = batch[0].to(self.device), batch[1].to(self.device)
                fused = local_model(mri, pet)
                loss = self._loss(loss_fn, fused, mri, pet, batch)
                opt.zero_grad()
                loss.backward()
                if self.mode == "dpsgd":
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), self.dp_clip)
                    for p in local_model.parameters():
                        if p.grad is not None:
                            p.grad.add_(torch.randn_like(p.grad) * self.dp_sigma * self.dp_clip)
                opt.step()

        local_state = local_model.state_dict()
        delta = {k: (local_state[k].detach() - self.global_state[k]).to(self.device) for k in self.global_state}
        return delta

    def _loss(self, loss_fn, fused, mri, pet, batch):
        if self.supervised_target:
            return loss_fn(fused, batch[2].to(self.device))
        return loss_fn(fused, mri, pet)

    # ------------------------------------------------------------------
    # Server-side aggregation
    # ------------------------------------------------------------------
    def _compress_delta(self, delta: dict, client_idx: int) -> tuple:
        """Return (payload, bytes_sent) for one client's update."""
        if self.mode == "fipca":
            flat = _flatten_state({k: delta[k] for k in self._param_keys}).to(self.device)
            if self.error_feedback:
                if self._ef_residuals[client_idx] is None:
                    self._ef_residuals[client_idx] = torch.zeros_like(flat)
                flat = flat + self._ef_residuals[client_idx]
            coeffs = self.P @ flat  # (rank,)
            if self.error_feedback:
                # What the server will reconstruct from this client's coeffs;
                # everything the projection dropped is carried to next round.
                recon = self.P.T @ coeffs
                self._ef_residuals[client_idx] = (flat - recon).detach()
            buffers = {k: delta[k].detach().clone() for k in self._buffer_keys}
            payload = (coeffs.detach().clone(), buffers)
            bytes_sent = coeffs.numel() * 4 + sum(v.numel() * v.element_size() for v in buffers.values())
            return payload, bytes_sent
        else:
            payload = {k: v.detach().clone() for k, v in delta.items()}
            bytes_sent = sum(v.numel() * 4 for v in payload.values())
            return payload, bytes_sent

    def _aggregate(self, client_payloads: list) -> dict:
        if self.mode == "fipca":
            avg_coeffs = torch.stack([cp[0] for cp in client_payloads]).mean(dim=0)  # (rank,)
            recon_flat = self.P.T @ avg_coeffs  # (n_params,)
            avg = _unflatten_to_state(recon_flat, {k: self.global_state[k] for k in self._param_keys})
            avg.update(self._mean_dicts([cp[1] for cp in client_payloads], self._buffer_keys))
            return avg
        return self._mean_dicts(client_payloads, list(self.global_state))

    @staticmethod
    def _mean_dicts(dicts: list, keys: list) -> dict:
        avg = {}
        for k in keys:
            stacked = torch.stack([cp[k] for cp in dicts])
            if stacked.dtype.is_floating_point or stacked.dtype.is_complex:
                avg[k] = stacked.mean(dim=0)
            else:
                # Integer buffers (e.g. BatchNorm.num_batches_tracked) cannot be
                # mean()-ed directly; promote to float, mean, and cast back.
                avg[k] = stacked.to(torch.float32).mean(dim=0).to(stacked.dtype)
        return avg

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    @torch.no_grad()
    def evaluate(self, ssim_score_fn, psnr_score_fn) -> dict:
        self.global_model.load_state_dict(self.global_state)
        self.global_model.eval()
        loss_fn = self.loss_fn_factory().to(self.device)
        losses, ssims, psnrs = [], [], []
        loader = DataLoader(self.eval_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)
        for batch in loader:
            mri, pet = batch[0].to(self.device), batch[1].to(self.device)
            fused = self.global_model(mri, pet)
            losses.append(self._loss(loss_fn, fused, mri, pet, batch).item())
            ref = batch[2].to(self.device) if self.supervised_target else 0.5 * (mri + pet)
            ssims.append(ssim_score_fn(fused, ref))
            psnrs.append(psnr_score_fn(fused, ref))
        return {
            "eval_loss": float(np.mean(losses)),
            "eval_ssim": float(np.mean(ssims)),
            "eval_psnr": float(np.mean(psnrs)),
        }

    # ------------------------------------------------------------------
    # Full federated run
    # ------------------------------------------------------------------
    def run(
        self,
        T: int,
        E: int,
        ssim_score_fn,
        psnr_score_fn,
        log_every: int = 5,
        checkpoint_every: int = 10,
        checkpoint_path: Optional[str] = None,
        tag: str = "run",
    ) -> list:
        K = len(self.client_datasets)
        history = []
        for t in range(1, T + 1):
            t0 = time.time()
            if self.mode == "fipca" and self.resample_basis:
                self._build_basis(self.seed + 7919 * t)
            payloads, total_bytes = [], 0
            for k in range(K):
                delta = self._train_client(k, E)
                payload, bytes_sent = self._compress_delta(delta, k)
                payloads.append(payload)
                total_bytes += bytes_sent
            avg_delta = self._aggregate(payloads)
            for k in self.global_state:
                self.global_state[k] = self.global_state[k] + avg_delta[k]
            self.global_model.load_state_dict(self.global_state)

            row = {
                "tag": tag,
                "round": t,
                "mode": self.mode,
                "round_secs": time.time() - t0,
                "bytes_per_round": int(total_bytes),
                "samples_per_local_epoch": self.samples_per_local_epoch,
                "fipca_rank": self.fipca_rank if self.mode == "fipca" else None,
                "error_feedback": self.error_feedback if self.mode == "fipca" else None,
                "resample_basis": self.resample_basis if self.mode == "fipca" else None,
                "dp_sigma": self.dp_sigma if self.mode == "dpsgd" else None,
            }
            if t % log_every == 0 or t == 1 or t == T:
                row.update(self.evaluate(ssim_score_fn, psnr_score_fn))
                print(
                    f"  [{tag}] r{t:02d}/{T} | "
                    f"loss={row['eval_loss']:.4f} ssim={row['eval_ssim']:.4f} "
                    f"psnr={row['eval_psnr']:.2f}dB | "
                    f"bytes={row['bytes_per_round']/1e6:.1f}MB/rd | "
                    f"{row['round_secs']:.1f}s"
                )
            if checkpoint_path and (t % checkpoint_every == 0 or t == T):
                torch.save(
                    {"state_dict": self.global_state, "round": t, "tag": tag},
                    f"{checkpoint_path}_r{t}.pt",
                )
            history.append(row)
        return history

In [ ]:
sys.path.insert(0, SRC_DIR)
import ifcnn, losses, trainer
for m in (ifcnn, losses, trainer):
    importlib.reload(m)
from ifcnn import IFCNN, InputMasked
from losses import TargetLoss, ssim_score, psnr_score
from trainer import FedAvgTrainer
def build_model():
    # IFCNN with output zeroed outside the input-derived brain mask (T1>0)|(T2>0)
    return InputMasked(IFCNN())
print('IFCNN params:', sum(p.numel() for p in build_model().parameters()))

In [ ]:
# 2. Partition + copy needed case files Drive -> local disk (resumable)
with open(f'{DRIVE_ROOT}/partitions/brats_K3_v2.json') as f:
    part = json.load(f)
clients = part['clients']
train_cases = [cl['train'] for cl in clients]
val_cases   = sorted(c for cl in clients for c in cl['val'])
test_cases  = part['test_cases']
needed = sorted({c for l in train_cases for c in l} | set(val_cases) | set(test_cases))
print(f'clients train patients={[len(l) for l in train_cases]} | val={len(val_cases)} | test={len(test_cases)}')

t0 = time.time(); copied = 0
for cid in needed:
    src, dst = f'{DRIVE_ROOT}/brats/processed_v2/{cid}.npz', f'{LOCAL}/{cid}.npz'
    if not os.path.exists(src):
        raise FileNotFoundError(f'{src} missing - re-run notebook 13')
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        continue
    shutil.copy(src, dst + '.tmp'); os.replace(dst + '.tmp', dst); copied += 1
print(f'Copied {copied} new files ({len(needed)} needed) in {time.time()-t0:.0f}s')

In [ ]:
# 3. Datasets: (t1, t2, flair) float32 tensors of shape (1,128,128); arrays kept in RAM as float16
from torch.utils.data import Dataset

def load_case(cid):
    with np.load(f'{LOCAL}/{cid}.npz') as d:
        return {k: d[k] for k in d.files}

class SliceDS(Dataset):
    def __init__(self, cases):
        parts = [load_case(c) for c in cases]
        self.t1 = np.concatenate([p['t1'] for p in parts])
        self.t2 = np.concatenate([p['t2'] for p in parts])
        self.fl = np.concatenate([p['flair'] for p in parts])
    def __len__(self):
        return len(self.t1)
    def __getitem__(self, i):
        f = lambda a: torch.from_numpy(a[i].astype(np.float32))[None]
        return f(self.t1), f(self.t2), f(self.fl)

t0 = time.time()
client_ds = [SliceDS(c) for c in train_cases]
pooled_ds = SliceDS([c for l in train_cases for c in l])
val_ds    = SliceDS(val_cases)
print(f'slices: clients={[len(d) for d in client_ds]} pooled={len(pooled_ds)} val={len(val_ds)} '
      f'| loaded in {time.time()-t0:.0f}s')
assert len(pooled_ds) == sum(len(d) for d in client_ds)

In [ ]:
# 4. Test-set metrics, identical definitions to notebook 13 cell 6 (+ brain-only PSNR/SSIM)
from skimage.metrics import structural_similarity as ssim_fn

@torch.no_grad()
def predict(model, d, bs=64):
    model.eval(); out = []
    for i in range(0, len(d['t1']), bs):
        a = torch.from_numpy(d['t1'][i:i+bs].astype(np.float32))[:, None].to(DEVICE)
        b = torch.from_numpy(d['t2'][i:i+bs].astype(np.float32))[:, None].to(DEVICE)
        out.append(model(a, b)[:, 0].float().cpu().numpy())
    return np.concatenate(out)

def case_metrics(p, d):
    fl = d['flair'].astype(np.float32); br = d['brain'] > 0
    maes, ssims, ssims_b, se_b, n_b = [], [], [], 0.0, 0
    for i in range(len(fl)):
        ssims.append(ssim_fn(fl[i], p[i], data_range=1.0))
        if br[i].any():
            maes.append(np.abs(p[i] - fl[i])[br[i]].mean())
            _, smap = ssim_fn(fl[i], p[i], data_range=1.0, full=True)
            ssims_b.append(smap[br[i]].mean())
            se_b += float(((p[i] - fl[i])[br[i]] ** 2).sum()); n_b += int(br[i].sum())
    mse = float(np.mean((p - fl) ** 2))
    return {'ssim': float(np.mean(ssims)), 'psnr': 10 * np.log10(1 / max(mse, 1e-12)),
            'mae_brain': float(np.mean(maes)), 'ssim_brain': float(np.mean(ssims_b)),
            'psnr_brain': 10 * np.log10(1 / max(se_b / max(n_b, 1), 1e-12))}

BASELINES = {
    'zero':        lambda t1, t2: np.zeros_like(t1),
    'T2':          lambda t1, t2: t2,
    'mean(T1,T2)': lambda t1, t2: 0.5 * (t1 + t2),
}
rows = []
for c in test_cases:
    d = load_case(c); t1, t2 = d['t1'].astype(np.float32), d['t2'].astype(np.float32)
    for name, f in BASELINES.items():
        rows.append({'case': c, 'method': name, **case_metrics(f(t1, t2), d)})
base_df = pd.DataFrame(rows)
base_sum = base_df.groupby('method').mean(numeric_only=True)
print(base_sum.round(4).to_string())

# Self-check: must reproduce notebook 13's numbers (same data, same metric code)
ref = pd.read_csv(f'{DRIVE_ROOT}/results/flair_trivial_baselines.csv', header=[0, 1], index_col=0)
for name in BASELINES:
    for m in ('ssim', 'psnr', 'mae_brain'):
        a, b = base_sum.loc[name, m], ref.loc[name, (m, 'mean')]
        assert abs(a - b) < 1e-3, f'metric mismatch {name}/{m}: {a:.4f} vs notebook-13 {b:.4f}'
print('Self-check vs notebook 13 baselines: PASSED')

In [ ]:
# 5. Run helpers (one (method, seed) run = train T rounds, save checkpoint, evaluate on test)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def run_paths(method, seed):
    tag = f'{method}_seed{seed}'
    return tag, f'{CKPT_DIR}/flair_{tag}.pt', f'{RUN_DIR}/{tag}_history.csv', f'{RUN_DIR}/{tag}_test.csv'

def run_done(method, seed):
    _, ck, hi, te = run_paths(method, seed)
    if not all(os.path.exists(p) for p in (ck, hi, te)):
        return False
    try:
        torch.load(ck, map_location='cpu'); pd.read_csv(hi); pd.read_csv(te)
        return True
    except Exception as e:
        print(f'  {method}/seed{seed}: unreadable output ({e}); will redo')
        return False

def make_trainer(method, seed, samples):
    set_seed(seed)
    ds = client_ds if method == 'fedavg' else [pooled_ds]
    per_client = samples if method == 'fedavg' else samples * len(client_ds)
    return FedAvgTrainer(
        build_model_fn=build_model, loss_fn_factory=lambda: TargetLoss(beta=1.0),
        client_datasets=ds, eval_dataset=val_ds, device=DEVICE, mode='standard',
        samples_per_local_epoch=per_client, lr=LR, batch_size=BATCH, seed=seed,
        supervised_target=True,
    )

def run_one(method, seed):
    tag, ck, hi, te = run_paths(method, seed)
    t0 = time.time()
    tr = make_trainer(method, seed, SAMPLES_PER_CLIENT)
    hist = tr.run(T=T_ROUNDS, E=E_LOCAL, ssim_score_fn=ssim_score, psnr_score_fn=psnr_score,
                  log_every=LOG_EVERY, tag=tag)
    model = build_model().to(DEVICE); model.load_state_dict(tr.global_state)
    rows = []
    for c in test_cases:
        d = load_case(c)
        rows.append({'case': c, 'method': method, 'seed': seed, **case_metrics(predict(model, d), d)})
    atomic_csv(pd.DataFrame(hist).assign(method=method, seed=seed), hi)
    atomic_csv(pd.DataFrame(rows), te)
    atomic_torch_save({'state_dict': tr.global_state, 'method': method, 'seed': seed, 'rounds': T_ROUNDS,
                       'samples_per_client': SAMPLES_PER_CLIENT, 'partition': 'brats_K3_v2'}, ck)  # last = done marker
    m = pd.DataFrame(rows).mean(numeric_only=True)
    print(f'  DONE {tag} in {(time.time()-t0)/60:.1f} min | test ssim={m.ssim:.4f} psnr={m.psnr:.2f} '
          f'mae_brain={m.mae_brain:.4f} ssim_brain={m.ssim_brain:.4f}')
print('OK')

In [ ]:
# 6. Sanity gate: short centralised run must beat mean(T1,T2) on validation before spending hours
if all(run_done(m, s) for m in METHODS for s in SEEDS):
    print('All runs already finished - gate skipped.')
else:
    t0 = time.time()
    gate = make_trainer('central', 0, SAMPLES_PER_CLIENT)
    gate.run(T=GATE_ROUNDS, E=1, ssim_score_fn=ssim_score, psnr_score_fn=psnr_score, log_every=GATE_ROUNDS, tag='gate')
    gm = build_model().to(DEVICE); gm.load_state_dict(gate.global_state)
    vm, vb = [], []
    for c in val_cases:
        d = load_case(c); p = predict(gm, d)
        vm.append(case_metrics(p, d)); vb.append(case_metrics(0.5 * (d['t1'].astype(np.float32) + d['t2'].astype(np.float32)), d))
    vm, vb = pd.DataFrame(vm).mean(), pd.DataFrame(vb).mean()
    print(f'Gate ({(time.time()-t0)/60:.1f} min) on VAL: model mae_brain={vm.mae_brain:.4f} ssim_brain={vm.ssim_brain:.4f} '
          f'| mean(T1,T2) mae_brain={vb.mae_brain:.4f} ssim_brain={vb.ssim_brain:.4f}')
    if vm.mae_brain >= vb.mae_brain:
        raise RuntimeError('Gate FAILED: model does not beat mean(T1,T2) after the short run. '
                           'Stop and paste this output to chat before running the long runs.')
    print('Gate PASSED')
    del gate, gm

In [ ]:
# 7. Main runs (resumable). Order: fedavg seed by seed, then central.
todo = [(m, s) for m in METHODS for s in SEEDS if not run_done(m, s)]
print(f'{len(todo)} run(s) to do:', todo)
for i, (m, s) in enumerate(todo):
    print(f'\n=== [{i+1}/{len(todo)}] {m} seed {s} ===')
    run_one(m, s)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('\nAll runs finished.')

In [ ]:
# 8. Summary: per-run means over the 54 test patients, then mean ± std over seeds; baselines alongside
te = pd.concat([pd.read_csv(run_paths(m, s)[3]) for m in METHODS for s in SEEDS])
hist = pd.concat([pd.read_csv(run_paths(m, s)[2]) for m in METHODS for s in SEEDS])
atomic_csv(hist, f'{DRIVE_ROOT}/results/flair_train_history.csv')

metrics = ['ssim', 'psnr', 'mae_brain', 'ssim_brain', 'psnr_brain']
per_run = te.groupby(['method', 'seed'])[metrics].mean()
model_sum = per_run.groupby('method').agg(['mean', 'std'])
base_rows = base_sum[metrics].copy()
base_rows.columns = pd.MultiIndex.from_product([metrics, ['mean']])
summary = pd.concat([model_sum, base_rows])
summary.columns = [f'{a}_{b}' for a, b in summary.columns]
summary.index.name = 'method'
summary.reset_index().to_csv(f'{DRIVE_ROOT}/results/flair_test_summary.csv', index=False)
print(f'Test set ({len(test_cases)} patients). Models: mean (std) over {len(SEEDS)} seeds of the per-run patient mean. '
      'Baselines: deterministic.')
print(summary.round(4).to_string())
print('\nPer-run means:')
print(per_run.round(4).to_string())

In [ ]:
# 9. Qualitative figure: FedAvg seed 42 on 3 test patients
ck = torch.load(run_paths('fedavg', SEEDS[0])[1], map_location=DEVICE)
model = build_model().to(DEVICE); model.load_state_dict(ck['state_dict'])
rng = np.random.default_rng(0)
n_show = min(3, len(test_cases))
fig, ax = plt.subplots(n_show, 5, figsize=(15, 3 * n_show), squeeze=False)
for r, c in enumerate(rng.choice(test_cases, size=n_show, replace=False)):
    d = load_case(c); i = len(d['t1']) // 2; p = predict(model, {k: d[k][i:i+1] for k in ('t1', 't2')})[0]
    fl = d['flair'][i].astype(np.float32)
    for j, (img, title, kw) in enumerate([
        (d['t1'][i], 'T1 (input)', dict(cmap='gray', vmin=0, vmax=1)),
        (d['t2'][i], 'T2 (input)', dict(cmap='gray', vmin=0, vmax=1)),
        (fl, 'FLAIR (target)', dict(cmap='gray', vmin=0, vmax=1)),
        (p, 'FedAvg prediction', dict(cmap='gray', vmin=0, vmax=1)),
        (np.abs(p - fl), '|error|', dict(cmap='magma', vmin=0, vmax=0.5)),
    ]):
        ax[r, j].imshow(np.asarray(img, dtype=np.float32), **kw); ax[r, j].axis('off')
        ax[r, j].set_title(f'{c[-3:]} {title}' if j == 0 else title, fontsize=9)
plt.tight_layout()
out = f'{DRIVE_ROOT}/figures/phase3_flair_examples.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show()
print('Saved', out)

## Done — paste back to chat
1. The **gate** line (cell 6).
2. The **summary table** and **per-run means** (cell 8).
3. Whether `phase3_flair_examples.png` looks sensible.

Then download `results/flair_test_summary.csv`, `results/flair_train_history.csv` and the PNG into the local repo.